In [1]:
!pip install -q langgraph langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 7.9 MB/s eta 0:00:00


In [3]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")


Enter your OpenAI API key: ··········


In [7]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# ==========================================
# STATE
# ==========================================

class AgentState(TypedDict):
    user_input: str
    classification: str
    route: str
    response: str


# ==========================================
# NODE 1: CLASSIFY
# ==========================================

def classify(state: AgentState):

    text = state["user_input"].lower()

    # Simple rule-based classification
    technical_words = [
        "crash", "error", "bug", "api",
        "login", "password", "broken",
        "not working", "upload", "server"
    ]

    billing_words = [
        "charge", "charged", "payment",
        "invoice", "refund", "subscription",
        "billing", "money", "price"
    ]

    if any(word in text for word in technical_words):
        classification = "technical"

    elif any(word in text for word in billing_words):
        classification = "billing"

    else:
        classification = "general"

    print(f"🔎 Classification: {classification}")

    return {
        "classification": classification
    }


# ==========================================
# NODE 2: ROUTE
# ==========================================

def route(state: AgentState):

    classification = state["classification"]

    if classification == "technical":
        selected_route = "technical_support"

    elif classification == "billing":
        selected_route = "billing_support"

    else:
        selected_route = "general_support"

    print(f"➡️ Route: {selected_route}")

    return {
        "route": selected_route
    }


# ==========================================
# NODE 3: RESPOND
# ==========================================

def respond(state: AgentState):

    route = state["route"]

    if route == "technical_support":

        response = (
            "Your issue has been routed to technical support. "
            "Please provide the error details so the technical team "
            "can investigate."
        )

    elif route == "billing_support":

        response = (
            "Your issue has been routed to billing support. "
            "The billing team can investigate charges, payments, "
            "refunds, or subscription issues."
        )

    else:

        response = (
            "Your request has been routed to general support. "
            "A support representative can help with your question."
        )

    return {
        "response": response
    }


# ==========================================
# BUILD GRAPH
# ==========================================

builder = StateGraph(AgentState)

builder.add_node("classify", classify)
builder.add_node("route", route)
builder.add_node("respond", respond)

builder.add_edge(START, "classify")
builder.add_edge("classify", "route")


# ==========================================
# CONDITIONAL EDGE
# ==========================================

def choose_route(state: AgentState):

    if state["classification"] == "technical":
        return "technical"

    elif state["classification"] == "billing":
        return "billing"

    else:
        return "general"


builder.add_conditional_edges(
    "route",
    choose_route,
    {
        "technical": "respond",
        "billing": "respond",
        "general": "respond"
    }
)

builder.add_edge("respond", END)

graph = builder.compile()

print("✅ LangGraph compiled successfully!")


✅ LangGraph compiled successfully!


In [8]:
test_inputs = [
    "My application keeps crashing when I upload a file.",
    "Why was I charged twice this month?",
    "What are your business hours?",
    "The API returns a 500 error when I send a request.",
    "I don't recognize this charge on my account."
]


expected_routes = [
    "technical_support",
    "billing_support",
    "general_support",
    "technical_support",
    "billing_support"
]


print("=" * 70)
print("TESTING 5 INPUTS")
print("=" * 70)

results = []

for i, user_input in enumerate(test_inputs):

    print(f"\n🧪 TEST {i + 1}")
    print(f"Input: {user_input}")

    result = graph.invoke({
        "user_input": user_input,
        "classification": "",
        "route": "",
        "response": ""
    })

    expected = expected_routes[i]
    actual = result["route"]

    correct = actual == expected

    print(f"Expected route: {expected}")
    print(f"Actual route:   {actual}")
    print(f"Routing correct: {'✅ YES' if correct else '❌ NO'}")

    print(f"\nResponse:")
    print(result["response"])

    results.append({
        "input": user_input,
        "classification": result["classification"],
        "route": actual,
        "expected_route": expected,
        "correct": correct
    })


TESTING 5 INPUTS

🧪 TEST 1
Input: My application keeps crashing when I upload a file.
🔎 Classification: technical
➡️ Route: technical_support
Expected route: technical_support
Actual route:   technical_support
Routing correct: ✅ YES

Response:
Your issue has been routed to technical support. Please provide the error details so the technical team can investigate.

🧪 TEST 2
Input: Why was I charged twice this month?
🔎 Classification: billing
➡️ Route: billing_support
Expected route: billing_support
Actual route:   billing_support
Routing correct: ✅ YES

Response:
Your issue has been routed to billing support. The billing team can investigate charges, payments, refunds, or subscription issues.

🧪 TEST 3
Input: What are your business hours?
🔎 Classification: general
➡️ Route: general_support
Expected route: general_support
Actual route:   general_support
Routing correct: ✅ YES

Response:
Your request has been routed to general support. A support representative can help with your question.


In [9]:
import pandas as pd

df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("ROUTING VERIFICATION")
print("=" * 70)

display(df)

total = len(df)
correct = df["correct"].sum()

print(f"\nCorrect: {correct}/{total}")

if correct == total:
    print("🎉 All 5 inputs were routed correctly!")
else:
    print("⚠️ Some inputs were not routed as expected.")



ROUTING VERIFICATION


,input,classification,route,expected_route,correct
0,My application keeps crashing when I upload a ...,technical,technical_support,technical_support,True
1,Why was I charged twice this month?,billing,billing_support,billing_support,True
2,What are your business hours?,general,general_support,general_support,True
3,The API returns a 500 error when I send a requ...,technical,technical_support,technical_support,True
4,I don't recognize this charge on my account.,billing,billing_support,billing_support,True



Correct: 5/5
🎉 All 5 inputs were routed correctly!


In [18]:
!pip install -q -U langgraph


In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
import os
from getpass import getpass


class HumanState(TypedDict):
    user_input: str
    classification: str
    route: str
    human_feedback: str
    response: str


# Ensure OPENAI_API_KEY is set
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=os.environ["OPENAI_API_KEY"]
)

Enter your OpenAI API key: ··········


In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    user_input: str
    classification: str
    route: str
    response: str


# -------- CLASSIFY --------

def classify(state):
    text = state["user_input"].lower()

    technical = [
        "crash", "error", "bug", "api",
        "upload", "server", "login",
        "not working", "password"
    ]

    billing = [
        "charge", "charged", "payment",
        "invoice", "refund", "subscription",
        "billing", "price"
    ]

    if any(word in text for word in technical):
        category = "technical"
    elif any(word in text for word in billing):
        category = "billing"
    else:
        category = "general"

    print("Classification:", category)

    return {"classification": category}


# -------- ROUTE --------

def route(state):
    category = state["classification"]

    if category == "technical":
        route_name = "technical_support"
    elif category == "billing":
        route_name = "billing_support"
    else:
        route_name = "general_support"

    print("Route:", route_name)

    return {"route": route_name}


# -------- RESPOND --------

def respond(state):
    route_name = state["route"]

    responses = {
        "technical_support":
            "Your request was sent to technical support.",

        "billing_support":
            "Your request was sent to billing support.",

        "general_support":
            "Your request was sent to general support."
    }

    return {
        "response": responses[route_name]
    }


# -------- BUILD GRAPH --------

builder = StateGraph(AgentState)

builder.add_node("classify", classify)
builder.add_node("route", route)
builder.add_node("respond", respond)

builder.add_edge(START, "classify")
builder.add_edge("classify", "route")


def choose_route(state):
    return state["classification"]


builder.add_conditional_edges(
    "route",
    choose_route,
    {
        "technical": "respond",
        "billing": "respond",
        "general": "respond"
    }
)

builder.add_edge("respond", END)

graph = builder.compile()

print("✅ Graph created successfully!")


✅ Graph created successfully!


In [3]:
def human_classify(state: HumanState):

    prompt = f"""
Classify this message into exactly one category:

technical
billing
general

Message:
{state["user_input"]}

Return only the category.
"""

    result = llm.invoke(prompt)

    classification = result.content.strip().lower()

    if classification not in ["technical", "billing", "general"]:
        classification = "general"

    print(f"🤖 AI Classification: {classification}")

    return {
        "classification": classification
    }


def human_route(state: HumanState):

    classification = state["classification"]

    if classification == "technical":
        route = "technical_support"
    elif classification == "billing":
        route = "billing_support"
    else:
        route = "general_support"

    print(f"🤖 AI Route: {route}")

    return {
        "route": route
    }


def human_review(state: HumanState):

    print("\n⏸️ GRAPH PAUSED")
    print("The AI wants to use this route:")
    print(f"Classification: {state['classification']}")
    print(f"Route: {state['route']}")

    feedback = interrupt({
        "message": "Human review required",
        "classification": state["classification"],
        "route": state["route"]
    })

    return {
        "human_feedback": feedback
    }


def human_respond(state: HumanState):

    prompt = f"""
Respond to the customer.

Customer:
{state["user_input"]}

AI classification:
{state["classification"]}

AI route:
{state["route"]}

Human reviewer feedback:
{state["human_feedback"]}

Use the human feedback when generating the response.
"""

    result = llm.invoke(prompt)

    return {
        "response": result.content
    }


In [12]:
human_builder = StateGraph(HumanState)

human_builder.add_node("classify", human_classify)
human_builder.add_node("route", human_route)
human_builder.add_node("human_review", human_review)
human_builder.add_node("respond", human_respond)

human_builder.add_edge(START, "classify")
human_builder.add_edge("classify", "route")
human_builder.add_edge("route", "human_review")
human_builder.add_edge("human_review", "respond")
human_builder.add_edge("respond", END)


# Memory is required so the graph can resume
memory = MemorySaver()

human_graph = human_builder.compile(
    checkpointer=memory
)

print("✅ Human-in-the-loop graph created!")


✅ Human-in-the-loop graph created!


In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver


# ============================================
# STATE
# ============================================

class AgentState(TypedDict):
    user_input: str
    classification: str
    route: str
    human_feedback: str
    response: str


# ============================================
# NODE 1: CLASSIFY
# ============================================

def classify(state: AgentState):

    text = state["user_input"].lower()

    technical_words = [
        "crash", "error", "bug", "api",
        "upload", "server", "login"
    ]

    billing_words = [
        "charge", "charged", "payment",
        "invoice", "refund", "subscription",
        "billing", "price"
    ]

    if any(word in text for word in technical_words):
        classification = "technical"

    elif any(word in text for word in billing_words):
        classification = "billing"

    else:
        classification = "general"

    print("🤖 Classification:", classification)

    return {
        "classification": classification
    }


# ============================================
# NODE 2: ROUTE
# ============================================

def route(state: AgentState):

    classification = state["classification"]

    if classification == "technical":
        selected_route = "technical_support"

    elif classification == "billing":
        selected_route = "billing_support"

    else:
        selected_route = "general_support"

    print("🤖 Route:", selected_route)

    return {
        "route": selected_route
    }


# ============================================
# NODE 3: HUMAN REVIEW
# ============================================

def human_review(state: AgentState):

    print("\n" + "=" * 50)
    print("⏸️ HUMAN REVIEW REQUIRED")
    print("=" * 50)

    print("User:", state["user_input"])
    print("AI Classification:", state["classification"])
    print("AI Route:", state["route"])

    # PAUSE GRAPH
    feedback = interrupt({
        "question": "Approve this route or provide a correction",
        "classification": state["classification"],
        "route": state["route"]
    })

    return {
        "human_feedback": feedback
    }


# ============================================
# NODE 4: RESPOND
# ============================================

def respond(state: AgentState):

    feedback = state["human_feedback"]

    response = (
        f"Request: {state['user_input']}\n"
        f"Classification: {state['classification']}\n"
        f"Route: {state['route']}\n"
        f"Human feedback: {feedback}\n\n"
        f"✅ Request processed after human approval."
    )

    return {
        "response": response
    }


# ============================================
# BUILD GRAPH
# ============================================

builder = StateGraph(AgentState)

builder.add_node("classify", classify)
builder.add_node("route", route)
builder.add_node("human_review", human_review)
builder.add_node("respond", respond)

builder.add_edge(START, "classify")
builder.add_edge("classify", "route")
builder.add_edge("route", "human_review")
builder.add_edge("human_review", "respond")
builder.add_edge("respond", END)


# ============================================
# CHECKPOINT MEMORY
# ============================================

memory = MemorySaver()

human_graph = builder.compile(
    checkpointer=memory
)

print("✅ Human-in-the-loop graph compiled!")


✅ Human-in-the-loop graph compiled!


In [5]:
config = {
    "configurable": {
        "thread_id": "human-test-001"
    }
}

initial_state = {
    "user_input": "I was charged twice for my subscription.",
    "classification": "",
    "route": "",
    "human_feedback": "",
    "response": ""
}

result = human_graph.invoke(
    initial_state,
    config=config
)

print("\nGraph paused.")
print(result)


🤖 Classification: billing
🤖 Route: billing_support

⏸️ HUMAN REVIEW REQUIRED
User: I was charged twice for my subscription.
AI Classification: billing
AI Route: billing_support

Graph paused.
{'user_input': 'I was charged twice for my subscription.', 'classification': 'billing', 'route': 'billing_support', 'human_feedback': '', 'response': '', '__interrupt__': [Interrupt(value={'question': 'Approve this route or provide a correction', 'classification': 'billing', 'route': 'billing_support'}, id='e4d82744c23be58d1ea577e5ffd10086')]}


In [6]:
human_feedback = """
Approved. The request is correctly classified as billing.
Please continue with billing support.
"""


In [7]:
result = human_graph.invoke(
    Command(resume=human_feedback),
    config=config
)

print("\n" + "=" * 50)
print("✅ GRAPH RESUMED")
print("=" * 50)

print(result["response"])



⏸️ HUMAN REVIEW REQUIRED
User: I was charged twice for my subscription.
AI Classification: billing
AI Route: billing_support

✅ GRAPH RESUMED
Request: I was charged twice for my subscription.
Classification: billing
Route: billing_support
Human feedback: 
Approved. The request is correctly classified as billing.
Please continue with billing support.


✅ Request processed after human approval.
